# 🛡️ IoT Device Identification : Hybrid Adversarial Training (CSV + JSON) — **SANS Borderline-SMOTE**

Ce notebook entraîne les modèles sur **DEUX datasets** (CSV + JSON) avec:
- **Dataset CSV** : IPFIX ML Instances (12 foyers, 18 classes IoT)
- **Dataset JSON** : IPFIX Records (3 mois, 17 classes IoT)
- **Curriculum Learning** (2 phases: Standard → Adversarial)
- **Early Stopping par phase**
- **Crash Test** après chaque phase (bénignes + adversaires + mélangé)
- **Macro F1-Score** et métriques par classe pour chaque phase
- **Plots de présentation** générés automatiquement à chaque phase
- **⚠️ SANS Borderline-SMOTE** — pour comparaison avec la version avec équilibrage

Tous les résultats et plots sont sauvegardés sur **Google Drive**.

# 🛠️ Phase 1 : Configuration de l'Environnement (Setup)

In [1]:
# ─── Cell 1: Setup ───────────────────────────────────────────────────────────
# Mount Google Drive FIRST so the paths in Cell 2 resolve correctly
from google.colab import drive
drive.mount('/content/drive')

# Clone the repository (pull latest if already cloned)
import os
if os.path.exists('/content/pfe'):
    !cd /content/pfe && git pull
else:
    !git clone https://github.com/yacinemkk/pfe.git /content/pfe

%cd /content/pfe

# Install dependencies
!pip install -q torch torchvision tqdm numpy pandas scikit-learn matplotlib xgboost psutil

Mounted at /content/drive
Cloning into '/content/pfe'...
remote: Enumerating objects: 597, done.
remote: Counting objects: 100% (320/320), done.
remote: Compressing objects: 100% (232/232), done.
remote: Total 597 (delta 140), reused 246 (delta 83), pack-reused 277 (from 1)
Receiving objects: 100% (597/597), 14.19 MiB | 18.44 MiB/s, done.
Resolving deltas: 100% (260/260), done.
/content/pfe


# ⚙️ Phase 2 : Configuration & Pipeline de Données

In [2]:
# ─── Cell 2: Configure ───────────────────────────────────────────────────────
# ⚠️  UPDATE THESE PATHS before pressing Run All.

# Path to your JSON IPFIX files on Google Drive.
# Expected structure:
#   JSON_DATA_DIR/
#     20-01-31(1)/ipfix_202001_fixed.json
#     20-03-31/ipfix_202003.json
#     20-04-30/ipfix_202004.json
JSON_DATA_DIR = '/content/drive/MyDrive/PFE/IPFIX_Records'

# Path to your CSV IPFIX ML Instances on Google Drive.
# Expected structure:
#   CSV_DATA_DIR/
#     home1_labeled.csv
#     home2_labeled.csv
#     ...
CSV_DATA_DIR = '/content/drive/MyDrive/PFE/IPFIX_ML_Instances'

# Where to save trained models, results.json, history.json, and plots.
# This folder is on Drive — it persists after the runtime disconnects.
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/PFE/results'

# Datasets to train on: 'csv', 'json', or 'both'
DATASETS = 'both'  # Options: 'csv', 'json', 'both'

# ─── Training hyperparameters ─────────────────────────────────────────────────
SEQ_LENGTH   = 10      # Sequence length (timesteps per sample)
EPOCHS       = 30      # Total epochs (split across 2 phases)
BATCH_SIZE   = 64      # Batch size
LEARNING_RATE = 1e-3   # Learning rate
ADV_METHOD   = 'hybrid' # Adversarial method: none, feature, pgd, fgsm, hybrid
ADV_RATIO    = 0.4     # Ratio of adversarial samples during training
MAX_FILES    = None    # Max CSV files to load (None = all). Use small int for testing.
MAX_RECORDS  = None    # Max JSON records to load (None = all). Use small int for testing.

# ─── Nouvelles Contre-Mesures Adversariales (5 Couches) ────────────────
PHASE2_EPOCHS    = 25      # Epochs Phase 2 (augmenté pour meilleure robustesse)
LABEL_SMOOTHING  = 0.1     # Label smoothing (nouveau)
USE_AFD          = True    # Couche 1: Adversarial Feature Dropout (nouveau)
AFD_P_SINGLE     = 0.3     # AFD: proba de dropper 1 feature
AFD_P_DOUBLE     = 0.15    # AFD: proba de dropper 2 features
AFD_P_MIMIC      = 0.1     # AFD: proba de mimic replacement
USE_DAE          = True    # Couche 3: Denoising Autoencoder (nouveau)
DAE_LATENT_DIM   = 16      # DAE: dimension latente
DAE_PRETRAIN     = 10      # DAE: epochs de pre-training
USE_RS           = True    # Couche 4: Randomized Smoothing (nouveau)
RS_SIGMA         = 0.25    # RS: bruit sigma
RS_N_SAMPLES     = 50      # RS: nombre d'echantillons bruites
LAMBDA_TRADES    = 3.0     # TRADES trade-off parameter (was 6.0)
TRADES_EPSILON   = 0.5     # Perturbation budget (was 0.05)
TRADES_PGD_STEPS = 15      # PGD steps (was 7)
USE_INPUT_TRANSFORM = True # Ajouter bruit gaussien + dropout
USE_CUTMIX       = True    # Adversarial CutMix


# ─── RAM Optimization ─────────────────────────────────────────────────────────
STRIDE = 10             # Larger stride = fewer sequences = less RAM
EVAL_SUBSAMPLE = 1000  # Max samples for adversarial eval
EVAL_BATCH_SIZE = 32   # Smaller batch for evaluation (saves RAM)
LEARNING_RATE = 1e-3   # Learning rate

# Create results directory if needed
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print(f'CSV data:     {CSV_DATA_DIR}')
print(f'JSON data:    {JSON_DATA_DIR}')
print(f'Results dir:  {DRIVE_RESULTS_DIR}')
print(f'Datasets:     {DATASETS}')
print(f'Seq length:   {SEQ_LENGTH}')
print(f'Epochs:       {EPOCHS} (Phase 1 + Phase 2)')
print(f'Batch size:   {BATCH_SIZE}')
print(f'Adv method:   {ADV_METHOD}')
print(f'Adv ratio:    {ADV_RATIO}')

# Verify CSV data exists
import glob
csv_files = glob.glob(f'{CSV_DATA_DIR}/home*_labeled.csv')
print(f'\nFound {len(csv_files)} CSV file(s):')
for f in sorted(csv_files)[:5]:
    size_mb = os.path.getsize(f) / (1024**2)
    print(f'  {os.path.basename(f)} ({size_mb:.1f} MB)')
if len(csv_files) > 5:
    print(f'  ... and {len(csv_files) - 5} more')

# Verify JSON data exists
json_files = glob.glob(f'{JSON_DATA_DIR}/**/*.json', recursive=True)
print(f'\nFound {len(json_files)} JSON file(s):')
for f in json_files:
    size_gb = os.path.getsize(f) / (1024**3)
    print(f'  {os.path.basename(f)} ({size_gb:.1f} GB)')

CSV data:     /content/drive/MyDrive/PFE/IPFIX_ML_Instances
JSON data:    /content/drive/MyDrive/PFE/IPFIX_Records
Results dir:  /content/drive/MyDrive/PFE/results
Datasets:     both
Seq length:   10
Epochs:       30 (Phase 1 + Phase 2)
Batch size:   64
Adv method:   hybrid
Adv ratio:    0.2

Found 12 CSV file(s):
  home10_labeled.csv (1099.6 MB)
  home11_labeled.csv (215.8 MB)
  home12_labeled.csv (332.2 MB)
  home1_labeled.csv (245.0 MB)
  home2_labeled.csv (297.0 MB)
  ... and 7 more

Found 3 JSON file(s):
  ipfix_202003.json (5.7 GB)
  ipfix_202004.json (5.3 GB)
  ipfix_202001_fixed.json (6.6 GB)


In [3]:
# ─── RAM Monitoring & Cleanup Utilities ───────────────────────────────────────
import gc
import psutil
import torch
import os

def get_memory_usage():
    process = psutil.Process(os.getpid())
    ram_gb = process.memory_info().rss / (1024**3)
    gpu_gb = 0
    if torch.cuda.is_available():
        gpu_gb = torch.cuda.memory_allocated() / (1024**3)
    return ram_gb, gpu_gb

def log_memory(label=''):
    ram_gb, gpu_gb = get_memory_usage()
    print(f'  [RAM {label}] {ram_gb:.2f} GB | [GPU {label}] {gpu_gb:.2f} GB')

def aggressive_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()
    ram_gb, gpu_gb = get_memory_usage()
    print(f'  [Cleanup] RAM: {ram_gb:.2f} GB | GPU: {gpu_gb:.2f} GB')

print('RAM monitoring utilities loaded.')
log_memory('startup')

RAM monitoring utilities loaded.
  [RAM startup] 0.56 GB | [GPU startup] 0.00 GB


# 🧠 Phase 3a : Entraînement sur CSV — Contre-Mesures Adversariales

**Chaque modèle est entraîné avec les contre-mesures suivantes:**
- **TRADES** (λ=3.0, ε=0.5) — Loss = CE + λ×KL(f(x)||f(x_adv))
- **Adversarial Feature Dropout (AFD)** — Zero-out aléatoire de features
- **Input Transform** — Bruit gaussien + dropout aléatoire
- **Denoising Autoencoder (DAE)** — Reconstruction de features clean
- **Adversarial CutMix** — Mélange clean/adversarial
- **Label Smoothing** (0.1) — Cibles moins extrêmes

**Workflow par modèle:**
- Phase 1: Entraînement standard (données bénignes)
- Phase 2: Entraînement adversarial (TRADES + Transform + CutMix)
- Évaluation finale: Clean + Adversarial + Robustness Ratio

**Données:** Chargées depuis Google Drive (preprocessed ou fresh loading)

**Modèles:** LSTM → BiLSTM → CNN-LSTM → XGBoost-LSTM → Transformer → CNN-BiLSTM-Transformer


## 📦 Chargement du Dataset CSV
Affiche les informations détaillées du dataset CSV avant l'entraînement.


In [4]:
def load_and_display_csv_dataset(csv_data_dir, seq_length=10, stride=10, save_dir=None):
    """Charge COMPLÈTEMENT le dataset CSV, affiche les infos, et sauvegarde sur Drive."""
    import sys
    import gc
    import numpy as np
    import pickle

    sys.path.insert(0, '/content/pfe')

    print("\n" + "=" * 70)
    print("  CHARGEMENT COMPLET DU DATASET CSV")
    print("=" * 70)

    print(f"\n  Répertoire : {csv_data_dir}")
    print(f"  Seq length : {seq_length} | Stride : {stride}")

    csv_files = sorted(glob.glob(f'{csv_data_dir}/home*_labeled.csv'))
    print(f"\n  Fichiers CSV trouvés : {len(csv_files)}")

    for f in csv_files:
        size_mb = os.path.getsize(f) / (1024**2)
        print(f"    {os.path.basename(f):<30s} : {size_mb:>10.1f} MB")

    total_gb = sum(os.path.getsize(f) for f in csv_files) / (1024**3)
    print(f"\n  Taille totale : {total_gb:.2f} GB")

    # Charger le dataset complet via le vrai pipeline CSV
    print("\n  Chargement via le pipeline CSV (IoTDataProcessor)...")
    from src.data.preprocessor import IoTDataProcessor

    processor = IoTDataProcessor()
    result = processor.process_all(
        max_files=None,
        data_dir=csv_data_dir,
        seq_length=seq_length,
        stride=stride,
        apply_balancing=False,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = len(features)

    print(f"\n  {'='*70}")
    print(f"  RÉSULTAT DU CHARGEMENT")
    print(f"  {'='*70}")
    print(f"    Features ({n_continuous}) : {features[:5]}...")
    print(f"    Classes ({len(label_encoder.classes_)}) : {list(label_encoder.classes_)}")
    print(f"\n  Shapes des séquences (seq_length={seq_length}, stride={stride}) :")
    print(f"    Train : {X_train.shape}  →  {len(X_train):,} séquences")
    print(f"    Val   : {X_val.shape}  →  {len(X_val):,} séquences")
    print(f"    Test  : {X_test.shape}  →  {len(X_test):,} séquences")
    print(f"    Total : {len(X_train) + len(X_val) + len(X_test):,} séquences")

    print(f"\n  Distribution des classes (train) :")
    for cls in label_encoder.classes_:
        cls_id = label_encoder.transform([cls])[0]
        count = int(np.sum(y_train == cls_id))
        bar = '█' * max(1, count // 50)
        print(f"    {cls:<30s} : {count:>6,}  {bar}")

    # ─── Sauvegarder sur Drive ────────────────────────────────────────────
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        print(f"\n  💾 Sauvegarde du dataset CSV pré-traité sur Drive...")
        print(f"     Répertoire : {save_dir}")

        np.save(f'{save_dir}/X_train.npy', X_train)
        np.save(f'{save_dir}/X_val.npy', X_val)
        np.save(f'{save_dir}/X_test.npy', X_test)
        np.save(f'{save_dir}/y_train.npy', y_train)
        np.save(f'{save_dir}/y_val.npy', y_val)
        np.save(f'{save_dir}/y_test.npy', y_test)

        with open(f'{save_dir}/csv_metadata.pkl', 'wb') as f:
            pickle.dump({
                'features': features,
                'scaler': scaler,
                'label_encoder': label_encoder,
                'n_continuous': n_continuous,
                'seq_length': seq_length,
                'stride': stride,
            }, f)

        # Marker file to indicate preprocessing is complete
        with open(f'{save_dir}/csv_ready', 'w') as f:
            f.write('ready')

        saved_gb = (X_train.nbytes + X_val.nbytes + X_test.nbytes +
                    y_train.nbytes + y_val.nbytes + y_test.nbytes) / (1024**3)
        print(f"  ✅ Dataset CSV sauvegardé ({saved_gb:.2f} GB)")
        print(f"     Fichiers : X_train, X_val, X_test, y_train, y_val, y_test, csv_metadata.pkl")

    print(f"\n  {'='*70}")
    print(f"  ✅ Dataset CSV chargé complètement en RAM")
    print(f"  {'='*70}\n")

    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }

# ─── CSV preprocessing directory on Drive ─────────────────────────────
CSV_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/csv'

if DATASETS in ['csv', 'both']:
    csv_data = load_and_display_csv_dataset(
        CSV_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
        save_dir=CSV_PREPROCESSED_DIR
    )
else:
    csv_data = None
    print('Skipping CSV dataset loading — DATASETS is not csv or both')



  CHARGEMENT COMPLET DU DATASET CSV

  Répertoire : /content/drive/MyDrive/PFE/IPFIX_ML_Instances
  Seq length : 10 | Stride : 10

  Fichiers CSV trouvés : 12
    home10_labeled.csv             :     1099.6 MB
    home11_labeled.csv             :      215.8 MB
    home12_labeled.csv             :      332.2 MB
    home1_labeled.csv              :      245.0 MB
    home2_labeled.csv              :      297.0 MB
    home3_labeled.csv              :      354.7 MB
    home4_labeled.csv              :     1541.7 MB
    home5_labeled.csv              :      366.6 MB
    home6_labeled.csv              :      932.0 MB
    home7_labeled.csv              :      527.0 MB
    home8_labeled.csv              :      317.2 MB
    home9_labeled.csv              :      238.2 MB

  Taille totale : 6.32 GB

  Chargement via le pipeline CSV (IoTDataProcessor)...
CSV-NATIVE IPFIX ML INSTANCES PREPROCESSOR (Pipeline 4 Etapes)

[ETAPE 1] Filtrage et adaptation au SDN...
  1.1 Chargement des fichiers CSV...
 

In [ ]:
# ─── Training with Countermeasures & Data Loading Helpers ────────────────────────
import sys
sys.path.insert(0, '/content/pfe')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
import gc
import os
import json
import pickle

from src.adversarial.trades import TRADESAttack, FeatureAttackGenerator, MultiAttackTRADES
from src.adversarial.input_transform import create_input_transform
from src.adversarial.cutmix import create_adversarial_cutmix
from src.adversarial.feature_dropout import AdversarialFeatureDropout, create_adversarial_feature_dropout
from src.adversarial.denoising_autoencoder import DenoisingAutoencoder, PerturbationGenerator, DenoisingAETrainer, create_denoising_autoencoder
from src.adversarial.attacks import SensitivityAnalysis, AdversarialSearch
from src.models.lstm import LSTMClassifier
from src.models.bilstm import BiLSTMClassifier
from src.models.cnn_lstm import CNNLSTMClassifier
from src.models.xgboost_lstm import XGBoostLSTMClassifier
from src.models.transformer import TransformerClassifier
from src.models.cnn_bilstm_transformer import CNNBiLSTMTransformerClassifier
from src.training.trainer import IoTSequenceDataset
from config.config import LSTM_CONFIG, TRANSFORMER_CONFIG, CNN_BILSTM_TRANSFORMER_CONFIG, LEARNING_RATE, WEIGHT_DECAY


def load_dataset_from_drive(dataset_type):
    """Load preprocessed data from Drive or fresh loading."""
    import numpy as np
    import pickle
    import os

    preprocessed_dir = f'{DRIVE_RESULTS_DIR}/preprocessed/{dataset_type}'
    ready_file = f'{preprocessed_dir}/{dataset_type}_ready'

    if os.path.exists(ready_file) and os.path.exists(f'{preprocessed_dir}/X_train.npy'):
        print(f'  📂 Loading preprocessed {dataset_type.upper()} data from Drive...')
        X_train = np.load(f'{preprocessed_dir}/X_train.npy')
        X_val = np.load(f'{preprocessed_dir}/X_val.npy')
        X_test = np.load(f'{preprocessed_dir}/X_test.npy')
        y_train = np.load(f'{preprocessed_dir}/y_train.npy')
        y_val = np.load(f'{preprocessed_dir}/y_val.npy')
        y_test = np.load(f'{preprocessed_dir}/y_test.npy')

        meta_file = f'{preprocessed_dir}/{dataset_type}_metadata.pkl'
        with open(meta_file, 'rb') as f:
            metadata = pickle.load(f)

        data = {
            'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
            'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
            'features': metadata['features'], 'scaler': metadata['scaler'],
            'label_encoder': metadata['label_encoder'], 'n_continuous': metadata['n_continuous']
        }
        print(f'  ✅ Preprocessed {dataset_type.upper()} data loaded: {len(X_train):,} train samples')
        return data
    else:
        print(f'  📂 No preprocessed data found. Loading {dataset_type.upper()} fresh...')
        if dataset_type == 'csv':
            return load_and_display_csv_dataset(
                CSV_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
                save_dir=CSV_PREPROCESSED_DIR
            )
        else:
            return load_and_display_json_dataset(
                JSON_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
                max_records=MAX_RECORDS, save_dir=JSON_PREPROCESSED_DIR
            )


def create_model(model_type, input_size, num_classes):
    """Create model instance based on model_type."""
    if model_type == 'lstm':
        return LSTMClassifier(input_size, num_classes)
    elif model_type == 'bilstm':
        return BiLSTMClassifier(input_size, num_classes)
    elif model_type == 'cnn_lstm':
        return CNNLSTMClassifier(input_size, num_classes)
    elif model_type == 'xgboost_lstm':
        return XGBoostLSTMClassifier(input_size, num_classes)
    elif model_type == 'transformer':
        return TransformerClassifier(input_size, num_classes)
    elif model_type == 'cnn_bilstm_transformer':
        return CNNBiLSTMTransformerClassifier(input_size, num_classes, seq_length=SEQ_LENGTH)
    else:
        raise ValueError(f"Model type {model_type} not supported")


def train_model_with_countermeasures(
    model_type,
    dataset_type,
    data_dict=None,
    lambda_trades=3.0,
    epsilon=0.5,
    alpha=0.05,
    pgd_steps=15,
    epochs_phase1=20,
    epochs_phase2=25,
    batch_size=64,
    lr=1e-3,
    use_input_transform=True,
    use_cutmix=True,
    label_smoothing=0.1,
    use_afd=True,
    afd_p_single=0.3,
    afd_p_double=0.15,
    afd_p_mimic=0.1,
    use_dae=True,
    dae_latent_dim=16,
    dae_pretrain_epochs=10,
    use_randomized_smoothing=True,
    rs_sigma=0.25,
    rs_n_samples=50,
    save_dir=None,
    use_class_weights=True,
):
    """
    Train a model with 5-layer countermeasures.
    Couche 1: AFD (Adversarial Feature Dropout)
    Couche 2: TRADES (corrected epsilon/lambda) + Label Smoothing
    Couche 3: DAE (Denoising Autoencoder)
    Couche 4: Randomized Smoothing (eval only)
    Phase 1: Standard training on benign data
    Phase 2: Adversarial training with all countermeasures
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*80}")
    print(f"  ENTRAÎNEMENT AVEC 5 COUCHES DE CONTRE-MESURES")
    print(f"{'='*80}")
    print(f"  Model: {model_type.upper()}")
    print(f"  Dataset: {dataset_type.upper()}")
    print(f"  Device: {device}")
    print(f"  Phase 1 epochs: {epochs_phase1}")
    print(f"  Phase 2 epochs: {epochs_phase2}")
    print(f"  Couche 1 - AFD: {use_afd}")
    print(f"  Couche 2 - TRADES: λ={lambda_trades}, ε={epsilon}, steps={pgd_steps}")
    print(f"  Label Smoothing: {label_smoothing}")
    print(f"  Couche 3 - DAE: {use_dae}")
    print(f"  Input Transform: {use_input_transform}")
    print(f"  Adversarial CutMix: {use_cutmix}")
    print(f"  Couche 4 - RS: {use_randomized_smoothing} (σ={rs_sigma})")
    print(f"{'='*80}\n")

    X_train = data_dict['X_train']
    X_val = data_dict['X_val']
    X_test = data_dict['X_test']
    y_train = data_dict['y_train']
    y_val = data_dict['y_val']
    y_test = data_dict['y_test']
    features = data_dict.get('features', [])
    n_continuous = data_dict.get('n_continuous', X_train.shape[2])
    label_encoder = data_dict.get('label_encoder', None)

    input_size = X_train.shape[2]
    num_classes = len(np.unique(y_train))

    print(f"  Input size: {input_size}")
    print(f"  Num classes: {num_classes}")
    print(f"  Train samples: {len(X_train):,}")
    print(f"  Val samples: {len(X_val):,}")
    print(f"  Test samples: {len(X_test):,}")

    train_dataset = IoTSequenceDataset(X_train, y_train)
    val_dataset = IoTSequenceDataset(X_val, y_val)
    test_dataset = IoTSequenceDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    model = create_model(model_type, input_size, num_classes)
    model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model parameters: {n_params:,}")

    # ── Sensitivity Analysis (for AFD + feature attacks) ──
    print(f"\n  Computing sensitivity analysis...")
    n_sens = min(5000, len(X_train))
    sens_indices = np.random.choice(len(X_train), n_sens, replace=False)
    sensitivity_analysis = SensitivityAnalysis(
        X_train.reshape(-1, X_train.shape[-1])[:n_sens*10],
        np.repeat(y_train[sens_indices], X_train.shape[1])[:n_sens*10],
        features if features else [f'f{i}' for i in range(input_size)],
        num_classes,
        n_continuous_features=n_continuous,
    )

    # ── PHASE 1: Standard Training ──
    print(f"\n{'─'*80}")
    print(f"  PHASE 1: ENTRAÎNEMENT STANDARD")
    print(f"{'─'*80}\n")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(epochs_phase1):
        model.train()
        train_loss = 0
        correct = 0
        total = 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                val_loss += criterion(outputs, y_batch).item()

        val_loss /= len(val_loader)
        train_acc = correct / total
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print(f"  Early stopping Phase 1 at epoch {epoch+1}")
                break

        print(f"  P1 Epoch {epoch+1}/{epochs_phase1} | Train Acc: {train_acc:.4f} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

    model.load_state_dict(best_state)
    model = model.to(device)
    print(f"  ✓ Phase 1 complete. Best val loss: {best_val_loss:.4f}")

    # ── PHASE 2: Adversarial Training with 5 Countermeasures ──
    print(f"\n{'─'*80}")
    print(f"  PHASE 2: ENTRAÎNEMENT ADVERSARIAL (5 Couches Contremesures)")
    print(f"{'─'*80}\n")

    # Couche 2: TRADES Attack (corrected epsilon)
    trades_attack = TRADESAttack(epsilon=epsilon, alpha=alpha, num_steps=pgd_steps)

    # Couche 1: AFD (Adversarial Feature Dropout)
    afd_layer = None
    if use_afd:
        non_mod_idx = []
        try:
            non_mod_idx = [sensitivity_analysis.feature_names.index(n) for n in sensitivity_analysis.non_modifiable if n in sensitivity_analysis.feature_names]
        except (ValueError, AttributeError):
            non_mod_idx = []
        afd_layer = create_adversarial_feature_dropout(
            n_features=len(sensitivity_analysis.feature_names),
            p_single=afd_p_single, p_double=afd_p_double, p_mimic=afd_p_mimic,
            non_modifiable_indices=non_mod_idx,
            benign_stats=sensitivity_analysis.benign_stats,
            n_continuous_features=n_continuous,
        ).to(device)
        print(f"  ✅ AFD layer created")

    # Couche 3: DAE (Denoising Autoencoder)
    dae = None
    dae_trainer = None
    if use_dae:
        non_mod_idx_ae = []
        try:
            non_mod_idx_ae = [sensitivity_analysis.feature_names.index(n) for n in sensitivity_analysis.non_modifiable if n in sensitivity_analysis.feature_names]
        except (ValueError, AttributeError):
            non_mod_idx_ae = []
        dae, pert_gen = create_denoising_autoencoder(
            input_size=input_size,
            latent_dim=dae_latent_dim,
            n_continuous_features=n_continuous,
            benign_stats=sensitivity_analysis.benign_stats,
            non_modifiable_indices=non_mod_idx_ae,
        )
        dae = dae.to(device)
        dae_trainer = DenoisingAETrainer(
            autoencoder=dae, classifier=model, device=device,
            perturbation_generator=pert_gen, alpha=1.0, beta=0.5,
        )
        print(f"  ✅ DAE created, pre-training...")
        try:
            dae_trainer.pretrain(train_loader, epochs=dae_pretrain_epochs, verbose=True)
            print(f"  ✅ DAE pre-training complete")
        except Exception as e:
            print(f"  ⚠️ DAE pre-training error: {e}. Continuing without DAE.")
            dae = None
            dae_trainer = None

    input_transform = None
    if use_input_transform:
        input_transform = create_input_transform(noise_std=0.02, dropout_prob=0.1, n_continuous_features=n_continuous)
        print(f"  Input Transform: noise_std=0.02, dropout=0.1")

    adv_cutmix = None
    if use_cutmix:
        adv_cutmix = create_adversarial_cutmix(alpha=1.0, prob=0.5)
        print(f"  Adversarial CutMix: α=1.0, prob=0.5")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr/2, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)

    best_val_loss = float('inf')
    best_state = None
    best_dae_state = None
    patience_counter = 0

    for epoch in range(epochs_phase2):
        model.train()
        if afd_layer is not None:
            afd_layer.train()
        train_loss = 0
        ce_loss_total = 0
        kl_loss_total = 0
        correct = 0
        total = 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # Couche 1: AFD
            if afd_layer is not None:
                X_batch = afd_layer(X_batch, y_batch)

            # Input Transform
            if input_transform:
                X_batch = input_transform(X_batch, training=True)

            # Couche 2: TRADES adversarial generation
            X_adv = trades_attack.generate(model, X_batch, y_batch, device)

            # CutMix
            if adv_cutmix and np.random.rand() < 0.5:
                X_adv, y_batch = adv_cutmix(X_batch, X_adv, y_batch)

            # Couche 3: DAE joint training step
            if dae_trainer is not None:
                try:
                    dae_trainer.joint_train_step(X_batch.clone(), y_batch.clone())
                except Exception:
                    pass

            optimizer.zero_grad()

            # Forward on clean (with AFD applied)
            if dae is not None:
                with torch.no_grad():
                    X_denoised = dae(X_batch)
                clean_logits = model(X_denoised)
            else:
                clean_logits = model(X_batch)
            ce_loss = criterion(clean_logits, y_batch)

            # Forward on adversarial
            if dae is not None:
                with torch.no_grad():
                    X_adv_denoised = dae(X_adv)
                adv_logits = model(X_adv_denoised)
            else:
                adv_logits = model(X_adv)

            # KL divergence
            clean_probs = F.softmax(clean_logits, dim=1).detach()
            adv_log_probs = F.log_softmax(adv_logits, dim=1)
            kl_loss = F.kl_div(adv_log_probs, clean_probs, reduction='batchmean')

            loss = ce_loss + lambda_trades * kl_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            ce_loss_total += ce_loss.item()
            kl_loss_total += kl_loss.item()
            _, predicted = clean_logits.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()

            del X_adv
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                val_loss += criterion(outputs, y_batch).item()

        val_loss /= len(val_loader)
        train_acc = correct / total
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if dae is not None:
                best_dae_state = {k: v.cpu().clone() for k, v in dae.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print(f"  Early stopping Phase 2 at epoch {epoch+1}")
                break

        avg_ce = ce_loss_total / len(train_loader)
        avg_kl = kl_loss_total / len(train_loader)
        print(f"  P2 Epoch {epoch+1}/{epochs_phase2} | Acc: {train_acc:.4f} | Loss: {train_loss/len(train_loader):.4f} | CE: {avg_ce:.4f} | KL: {avg_kl:.4f} | Val: {val_loss:.4f}")

    model.load_state_dict(best_state)
    model = model.to(device)
    if dae is not None and best_dae_state is not None:
        dae.load_state_dict(best_dae_state)
        dae = dae.to(device)
    print(f"  ✓ Phase 2 complete. Best val loss: {best_val_loss:.4f}")

    # ── Final Evaluation ──
    print(f"\n{'='*80}")
    print(f"  ÉVALUATION FINALE")
    print(f"{'='*80}\n")

    model.eval()
    correct_clean = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            if dae is not None:
                X_batch = dae(X_batch)
            outputs = model(X_batch)
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct_clean += predicted.eq(y_batch).sum().item()
    clean_acc = correct_clean / total
    print(f"  Clean Accuracy: {clean_acc:.4f}")

    correct_adv = 0
    total = 0
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        X_adv = trades_attack.generate(model, X_batch, y_batch, device)
        if dae is not None:
            with torch.no_grad():
                X_adv = dae(X_adv)
        with torch.no_grad():
            outputs = model(X_adv)
        _, predicted = outputs.max(1)
        total += y_batch.size(0)
        correct_adv += predicted.eq(y_batch).sum().item()
    adv_acc = correct_adv / total
    robustness_ratio = adv_acc / max(clean_acc, 1e-8)
    print(f"  Adversarial Accuracy: {adv_acc:.4f}")
    print(f"  Robustness Ratio: {robustness_ratio:.4f}")

    # Couche 4: Randomized Smoothing evaluation
    rs_results = None
    if use_randomized_smoothing:
        from src.adversarial.randomized_smoothing import RandomizedSmoothing
        print(f"\n  🛡️  Couche 4: Randomized Smoothing evaluation (σ={rs_sigma}, N={rs_n_samples})...")
        smoother = RandomizedSmoothing(model, device, sigma=rs_sigma, n_samples=rs_n_samples)
        rs_results = smoother.evaluate_robust(X_test, y_test)
        print(f"    RS Clean Accuracy: {rs_results['clean_accuracy']:.4f}")
        print(f"    Mean Certified Radius: {rs_results['mean_certified_radius']:.4f}")
        print(f"    % Certified: {rs_results['pct_certified']:.4f}")

    results = {
        'model_type': model_type,
        'dataset_type': dataset_type,
        'clean_accuracy': clean_acc,
        'adversarial_accuracy': adv_acc,
        'robustness_ratio': robustness_ratio,
        'lambda_trades': lambda_trades,
        'epsilon': epsilon,
        'label_smoothing': label_smoothing,
        'use_afd': use_afd,
        'use_dae': use_dae,
        'use_randomized_smoothing': use_randomized_smoothing,
        'epochs_phase1': epochs_phase1,
        'epochs_phase2': epochs_phase2,
        'use_input_transform': use_input_transform,
        'use_cutmix': use_cutmix,
    }
    if rs_results is not None:
        results['rs_clean_accuracy'] = rs_results['clean_accuracy']
        results['rs_mean_certified_radius'] = rs_results['mean_certified_radius']
        results['rs_pct_certified'] = rs_results['pct_certified']

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        torch.save(model.state_dict(), f'{save_dir}/model.pt')
        if dae is not None:
            torch.save(dae.state_dict(), f'{save_dir}/dae_model.pt')
        with open(f'{save_dir}/countermeasures_results.json', 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\n  💾 Model and results saved to: {save_dir}")

    del train_loader, val_loader, test_loader
    del train_dataset, val_dataset, test_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

print('✅ Training functions loaded: train_model_with_countermeasures, load_dataset_from_drive, create_model')


In [ ]:
# ─── MODEL: LSTM on CSV (with Countermeasures) ────────────────────────────
MODEL = 'lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — LSTM with Countermeasures on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_csv',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on CSV complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_csv')
        aggressive_cleanup()
    else:
        print('❌ Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


In [ ]:
# ─── MODEL: BILSTM on CSV (with Countermeasures) ────────────────────────────
MODEL = 'bilstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — BILSTM with Countermeasures on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_csv',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on CSV complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_csv')
        aggressive_cleanup()
    else:
        print('❌ Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


In [ ]:
# ─── MODEL: CNN-LSTM on CSV (with Countermeasures) ────────────────────────────
MODEL = 'cnn_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — CNN-LSTM with Countermeasures on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_csv',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on CSV complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_csv')
        aggressive_cleanup()
    else:
        print('❌ Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


In [ ]:
# ─── MODEL: XGBOOST-LSTM on CSV (with Countermeasures) ────────────────────────────
MODEL = 'xgboost_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — XGBOOST-LSTM with Countermeasures on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_csv',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on CSV complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_csv')
        aggressive_cleanup()
    else:
        print('❌ Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


In [ ]:
# ─── MODEL: TRANSFORMER on CSV (with Countermeasures) ────────────────────────────
MODEL = 'transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — TRANSFORMER with Countermeasures on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_csv',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on CSV complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_csv')
        aggressive_cleanup()
    else:
        print('❌ Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


In [ ]:
# ─── MODEL: CNN-BILSTM-TRANSFORMER on CSV (with Countermeasures) ────────────────────────────
MODEL = 'cnn_bilstm_transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3a — CNN-BILSTM-TRANSFORMER with Countermeasures on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_csv',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on CSV complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_csv')
        aggressive_cleanup()
    else:
        print('❌ Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n✅ {MODEL.upper()} on CSV DONE')


# 🧹 Nettoyage RAM entre CSV et JSON

Exécutez cette cellule une seule fois après avoir terminé tous les modèles CSV.

In [ ]:
# ─── CLEANUP RAM BEFORE JSON PHASE ──────────────────────────────────────
print('\n' + '='*80)
print('  🧹 CLEANING RAM BEFORE JSON PHASE...')
print('='*80 + '\n')

aggressive_cleanup()

print('\n✅ RAM cleaned. Ready for JSON phase.')


# 🧠 Phase 3b : Entraînement sur JSON — Contre-Mesures Adversariales

**Même méthode d'entraînement que Phase 3a (TRADES + Transform + CutMix)**
appliquée au dataset JSON avec labelisation par adresses MAC.


## 📦 Chargement du Dataset JSON
Affiche les informations détaillées du dataset JSON chargé avant l'entraînement.


In [ ]:
def load_and_display_json_dataset(json_data_dir, seq_length=10, stride=10, max_records=None, save_dir=None):
    """Charge COMPLÈTEMENT le dataset JSON, affiche les infos, et sauvegarde sur Drive."""
    import sys
    import gc
    import numpy as np
    import pickle

    sys.path.insert(0, '/content/pfe')

    print("\n" + "=" * 70)
    print("  CHARGEMENT COMPLET DU DATASET JSON")
    print("=" * 70)

    print(f"\n  Répertoire : {json_data_dir}")
    print(f"  Seq length : {seq_length} | Stride : {stride}")
    print(f"  Max records: {max_records if max_records else 'Tous'}")

    json_files = sorted(glob.glob(f'{json_data_dir}/**/*.json', recursive=True))
    print(f"\n  Fichiers JSON trouvés : {len(json_files)}")

    for f in json_files:
        size_gb = os.path.getsize(f) / (1024**3)
        print(f"    {os.path.basename(f):<30s} : {size_gb:>10.2f} GB")

    total_gb = sum(os.path.getsize(f) for f in json_files) / (1024**3)
    print(f"\n  Taille totale : {total_gb:.2f} GB")

    # Charger le dataset complet via le vrai pipeline JSON
    print("\n  Chargement via le pipeline JSON (JsonIoTDataProcessor)...")
    from src.data.json_preprocessor import JsonIoTDataProcessor

    processor = JsonIoTDataProcessor()
    result = processor.process_all(
        data_dir=json_data_dir,
        seq_length=seq_length,
        stride=stride,
        max_records=max_records,
        apply_balancing=False,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = 36  # JSON pipeline features

    print(f"\n  {'='*70}")
    print(f"  RÉSULTAT DU CHARGEMENT")
    print(f"  {'='*70}")
    print(f"    Features ({len(features)}) : {features[:5]}...")
    print(f"    Classes ({len(label_encoder.classes_)}) : {list(label_encoder.classes_)}")
    print(f"\n  Shapes des séquences (seq_length={seq_length}, stride={stride}) :")
    print(f"    Train : {X_train.shape}  →  {len(X_train):,} séquences")
    print(f"    Val   : {X_val.shape}  →  {len(X_val):,} séquences")
    print(f"    Test  : {X_test.shape}  →  {len(X_test):,} séquences")
    print(f"    Total : {len(X_train) + len(X_val) + len(X_test):,} séquences")

    print(f"\n  Distribution des classes (train) :")
    for cls in label_encoder.classes_:
        cls_id = label_encoder.transform([cls])[0]
        count = int(np.sum(y_train == cls_id))
        bar = '█' * max(1, count // 50)
        print(f"    {cls:<30s} : {count:>6,}  {bar}")

    # ─── Sauvegarder sur Drive ────────────────────────────────────────────
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        print(f"\n  💾 Sauvegarde du dataset JSON pré-traité sur Drive...")
        print(f"     Répertoire : {save_dir}")

        np.save(f'{save_dir}/X_train.npy', X_train)
        np.save(f'{save_dir}/X_val.npy', X_val)
        np.save(f'{save_dir}/X_test.npy', X_test)
        np.save(f'{save_dir}/y_train.npy', y_train)
        np.save(f'{save_dir}/y_val.npy', y_val)
        np.save(f'{save_dir}/y_test.npy', y_test)

        with open(f'{save_dir}/json_metadata.pkl', 'wb') as f:
            pickle.dump({
                'features': features,
                'scaler': scaler,
                'label_encoder': label_encoder,
                'n_continuous': n_continuous,
                'seq_length': seq_length,
                'stride': stride,
            }, f)

        # Marker file to indicate preprocessing is complete
        with open(f'{save_dir}/json_ready', 'w') as f:
            f.write('ready')

        saved_gb = (X_train.nbytes + X_val.nbytes + X_test.nbytes +
                    y_train.nbytes + y_val.nbytes + y_test.nbytes) / (1024**3)
        print(f"  ✅ Dataset JSON sauvegardé ({saved_gb:.2f} GB)")
        print(f"     Fichiers : X_train, X_val, X_test, y_train, y_val, y_test, json_metadata.pkl")

    print(f"\n  {'='*70}")
    print(f"  ✅ Dataset JSON chargé complètement en RAM")
    print(f"  {'='*70}\n")

    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }

# ─── JSON preprocessing directory on Drive ─────────────────────────────
JSON_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/json'

if DATASETS in ['json', 'both']:
    json_data = load_and_display_json_dataset(
        JSON_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
        max_records=MAX_RECORDS, save_dir=JSON_PREPROCESSED_DIR
    )
else:
    json_data = None
    print('Skipping JSON dataset loading — DATASETS is not json or both')


In [ ]:
# ─── MODEL: LSTM on JSON (with Countermeasures) ────────────────────────────
MODEL = 'lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — LSTM with Countermeasures on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_json',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on JSON complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_json')
        aggressive_cleanup()
    else:
        print('❌ Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL: BILSTM on JSON (with Countermeasures) ────────────────────────────
MODEL = 'bilstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — BILSTM with Countermeasures on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_json',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on JSON complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_json')
        aggressive_cleanup()
    else:
        print('❌ Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL: CNN-LSTM on JSON (with Countermeasures) ────────────────────────────
MODEL = 'cnn_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — CNN-LSTM with Countermeasures on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_json',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on JSON complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_json')
        aggressive_cleanup()
    else:
        print('❌ Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL: XGBOOST-LSTM on JSON (with Countermeasures) ────────────────────────────
MODEL = 'xgboost_lstm'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — XGBOOST-LSTM with Countermeasures on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_json',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on JSON complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_json')
        aggressive_cleanup()
    else:
        print('❌ Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL: TRANSFORMER on JSON (with Countermeasures) ────────────────────────────
MODEL = 'transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — TRANSFORMER with Countermeasures on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_json',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on JSON complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_json')
        aggressive_cleanup()
    else:
        print('❌ Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n✅ {MODEL.upper()} on JSON DONE')


In [ ]:
# ─── MODEL: CNN-BILSTM-TRANSFORMER on JSON (with Countermeasures) ────────────────────────────
MODEL = 'cnn_bilstm_transformer'
print(f'\n{"#"*80}')
print(f'  PHASE 3b — CNN-BILSTM-TRANSFORMER with Countermeasures on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')

    if data is not None:
        results = train_model_with_countermeasures(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            lambda_trades=LAMBDA_TRADES,
            epsilon=TRADES_EPSILON,
            alpha=TRADES_EPSILON / 10,
            pgd_steps=TRADES_PGD_STEPS,
            epochs_phase1=20,
            epochs_phase2=PHASE2_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            use_input_transform=USE_INPUT_TRANSFORM,
            use_cutmix=USE_CUTMIX,
            label_smoothing=LABEL_SMOOTHING,
            use_afd=USE_AFD,
            afd_p_single=AFD_P_SINGLE,
            afd_p_double=AFD_P_DOUBLE,
            afd_p_mimic=AFD_P_MIMIC,
            use_dae=USE_DAE,
            dae_latent_dim=DAE_LATENT_DIM,
            dae_pretrain_epochs=DAE_PRETRAIN,
            use_randomized_smoothing=USE_RS,
            rs_sigma=RS_SIGMA,
            rs_n_samples=RS_N_SAMPLES,
            save_dir=f'{DRIVE_RESULTS_DIR}/models/{MODEL}_countermeasures_json',
            use_class_weights=True,
        )

        print(f"\n✅ {MODEL.upper()} with Countermeasures on JSON complete!")
        print(f"  Clean Accuracy: {results['clean_accuracy']:.4f}")
        print(f"  Adversarial Accuracy: {results['adversarial_accuracy']:.4f}")
        print(f"  Robustness Ratio: {results['robustness_ratio']:.4f}")

        log_memory(f'after_{MODEL}_json')
        aggressive_cleanup()
    else:
        print('❌ Failed to load JSON data')
else:
    print('Skipping JSON dataset')



print(f'\n{"="*80}')
print('  ✅ ALL 6 MODELS TRAINED ON BOTH DATASETS — COMPLETE!')
print(f'{"="*80}')


# 📊 Phase 4 : Visualisation de TOUS les Résultats

Affiche les plots générés pendant l'entraînement — directement depuis Drive.

Les fichiers suivants sont générés pour chaque modèle, chaque dataset et chaque phase :
- `fig_device_identification_scores_phaseN.png` — P/R/F1 par appareil
- `fig_adversarial_effect_phaseN.png` — Impact des attaques par appareil
- `fig_robustness_summary_phaseN.png` — Accuracy + Macro-F1 clean vs attaques
- `fig_training_history.png` — Courbes loss/accuracy

In [ ]:
# ─── Cell: Display ALL Plots ─────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from IPython.display import display, HTML

results_dir = Path(DRIVE_RESULTS_DIR) / 'models'
if not results_dir.exists():
    print('No results found yet — run the training cells first.')
else:
    result_dirs = sorted(results_dir.iterdir())

    for rd in result_dirs:
        if not rd.is_dir():
            continue

        model_name = rd.name
        print(f"\n{'='*80}")
        print(f"  📁 {model_name}")
        print(f"{'='*80}")

        # Show results summary
        results_file = rd / 'results.json'
        if results_file.exists():
            with open(results_file) as f:
                res = json.load(f)
            print(f"  Model: {res.get('model_type', '?')} | Input: {res.get('input_size', '?')} | Classes: {res.get('num_classes', '?')}")
            print(f"  Clean Accuracy: {res.get('test_accuracy_clean', 0):.4f}")
            if 'clean_metrics' in res:
                cm = res['clean_metrics']
                print(f"  Macro F1: {cm.get('macro_f1', 0):.4f} | Macro P: {cm.get('macro_precision', 0):.4f} | Macro R: {cm.get('macro_recall', 0):.4f}")
            if 'robustness_ratios' in res:
                print(f"  Robustness Ratios: {res['robustness_ratios']}")

        # Display all plots
        plot_files = sorted(rd.glob('fig_*.png'))
        if plot_files:
            for pf in plot_files:
                print(f"\n  📊 {pf.name}")
                fig, ax = plt.subplots(figsize=(16, 8))
                img = mpimg.imread(str(pf))
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(pf.stem.replace('_', ' ').title(), fontsize=14, fontweight='bold')
                plt.tight_layout()
                plt.show()
        else:
            print('  No plots found.')

        # Show curriculum report if available
        report_file = rd / 'curriculum_report.json'
        if report_file.exists():
            with open(report_file) as f:
                report = json.load(f)
            print(f"\n  📋 Curriculum Report:")
            for phase_key, phase_data in report.get('phases', {}).items():
                phase_num = phase_data.get('phase', '?')
                clean = phase_data.get('clean', {})
                feat = phase_data.get('feature_attack', {})
                pgd = phase_data.get('sequence_pgd', {})
                fgsm = phase_data.get('sequence_fgsm', {})
                print(f"    Phase {phase_num}:")
                print(f"      Clean  — Acc: {clean.get('accuracy', 0):.4f}  F1: {clean.get('f1_score', 0):.4f}  Macro-F1: {clean.get('macro_f1', 0):.4f}")
                if feat:
                    print(f"      FeatAdv — Acc: {feat.get('accuracy', 0):.4f}  F1: {feat.get('f1_score', 0):.4f}  RR: {feat.get('robustness_ratio', 0):.4f}")
                if pgd:
                    print(f"      SeqPGD — Acc: {pgd.get('accuracy', 0):.4f}  F1: {pgd.get('f1_score', 0):.4f}  RR: {pgd.get('robustness_ratio', 0):.4f}")
                if fgsm:
                    print(f"      SeqFGSM — Acc: {fgsm.get('accuracy', 0):.4f}  F1: {fgsm.get('f1_score', 0):.4f}  RR: {fgsm.get('robustness_ratio', 0):.4f}")

print('\n\n✅ All results displayed.')

# 📋 Phase 5 : Tableau Récapitulatif Multi-Modèles (CSV + JSON)

Compare les performances de tous les modèles entraînés sur les deux datasets.

In [ ]:
# ─── Cell: Comparative Summary Table ─────────────────────────────────────────
import json
from pathlib import Path

results_dir = Path(DRIVE_RESULTS_DIR) / 'models'
if not results_dir.exists():
    print('No results found.')
else:
    print(f"{'Model':<30} {'Clean Acc':>10} {'Macro F1':>10} {'Feat Adv':>10} {'Seq PGD':>10} {'Seq FGSM':>10} {'Hybrid':>10}")
    print(f"{'-'*100}")

    for rd in sorted(results_dir.iterdir()):
        rf = rd / 'results.json'
        if not rf.exists():
            continue
        with open(rf) as f:
            res = json.load(f)

        model = rd.name
        clean_acc = res.get('test_accuracy_clean', 0)
        macro_f1 = res.get('clean_metrics', {}).get('macro_f1', 0)

        adv = res.get('adversarial_results', {})
        feat_acc = adv.get('feature_level', {}).get('accuracy', 0)
        pgd_acc = adv.get('sequence_pgd', {}).get('accuracy', 0)
        fgsm_acc = adv.get('sequence_fgsm', {}).get('accuracy', 0)
        hybrid_acc = adv.get('hybrid', {}).get('accuracy', 0)

        print(f"{model:<30} {clean_acc:>10.4f} {macro_f1:>10.4f} {feat_acc:>10.4f} {pgd_acc:>10.4f} {fgsm_acc:>10.4f} {hybrid_acc:>10.4f}")

    print(f"{'-'*100}")
    print('\n✅ Comparison complete.')

In [ ]:
# ─── Git Push ──────────────────────────────────────────────────────────────────
import subprocess

print('Pushing to GitHub...')
result = subprocess.run(['git', 'add', '-A'], capture_output=True, text=True, cwd='/content/pfe')
print(f'  git add: {result.returncode}')

result = subprocess.run(['git', 'commit', '-m', 'Update models with countermeasures training'], capture_output=True, text=True, cwd='/content/pfe')
if result.returncode == 0:
    print(f'  git commit: OK')
else:
    print(f'  git commit: {result.stdout.strip()} {result.stderr.strip()}')

result = subprocess.run(['git', 'push'], capture_output=True, text=True, cwd='/content/pfe')
if result.returncode == 0:
    print('  ✅ Push successful!')
else:
    print(f'  git push stderr: {result.stderr.strip()[:500]}')
